In [1]:
# %% [markdown]
# # Notebook 04 — Intégration Blockchain
#
# **Objectif** : Valider toutes les interactions Web3.py avec
# le smart contract MusicRegistry déployé sur Ganache.
#
# **Prérequis** :
# - Ganache démarré sur le port 7545
# - `blockchain/artifacts/deployment.json` généré (étape 1)
# - Clé privée Ganache dans `.env` (DEPLOYER_PRIVATE_KEY)

# %%
import sys, os, time, json
import numpy as np

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from web3 import Web3
from backend.core.blockchain_manager import BlockchainManager
from backend.core.fingerprint_engine import FingerprintEngine
from backend.config                  import settings

print(f"Ganache URL : {settings.ganache_url}")

# %% [markdown]
# ## 1. Connexion à Ganache

# %%
manager = BlockchainManager()

t0 = time.time()
manager.connect_web3()
dt = time.time() - t0

print(f"Connexion établie en : {dt:.3f}s")
print(f"Connecté             : {'✅' if manager.is_connected() else '❌'}")

w3 = Web3(Web3.HTTPProvider(settings.ganache_url))
print(f"Block courant        : {w3.eth.block_number}")
print(f"Comptes Ganache      : {len(w3.eth.accounts)}")
print(f"Solde déployeur      : {w3.from_wei(w3.eth.get_balance(w3.eth.accounts[0]), 'ether'):.4f} ETH")

Ganache URL : http://127.0.0.1:7545
Connexion établie en : 0.220s
Connecté             : ✅
Block courant        : 1
Comptes Ganache      : 10
Solde déployeur      : 999.9988 ETH


In [2]:
# %% [markdown]
# ## 2. Test registerWork()

# %%
# Génération d'empreintes de test
engine = FingerprintEngine()
engine.load_grafprint_model()

import soundfile as sf, tempfile

def make_audio_file(freq=440.0, sr=8000, duration=5.0):
    t = np.linspace(0, duration, int(sr * duration))
    s = (np.sin(2 * np.pi * freq * t) * 0.5).astype(np.float32)
    f = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(f.name, s, sr)
    return f.name

# Œuvre 1 : Artiste A (70%) + Producteur B (30%)
audio1 = make_audio_file(freq=440.0)
emb1   = engine.extract_fingerprint(audio1)
hash1  = FingerprintEngine.embedding_to_hash(emb1)

accounts    = w3.eth.accounts
recipient1  = accounts[1]
recipient2  = accounts[2]
cid_test    = "QmTestFingerprint001AAABBBCCC"

print("Test registerWork()")
print("─" * 50)
print(f"  Work hash   : {hash1[:32]}...")
print(f"  CID IPFS    : {cid_test}")
print(f"  Artiste     : {recipient1}  (70%)")
print(f"  Producteur  : {recipient2}  (30%)")

t0 = time.time()
tx_hash = manager.register_work(
    work_hash  = hash1,
    cid        = cid_test,
    recipients = [recipient1, recipient2],
    shares     = [70, 30]
)
dt = time.time() - t0

print(f"\n  tx_hash  : {tx_hash[:32]}...")
print(f"  Temps    : {dt:.3f}s")
assert tx_hash.startswith("0x"), "tx_hash doit commencer par 0x"
print("  ✅ registerWork() validé")

c:\Python311\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Test registerWork()
──────────────────────────────────────────────────
  Work hash   : 935856c3274fcd604dc5f89b54a197f5...
  CID IPFS    : QmTestFingerprint001AAABBBCCC
  Artiste     : 0x6217386b3d46D78beb721FC3be503f46D2008689  (70%)
  Producteur  : 0xF046B0CBce9EdD632Af48768559DeEB46E2701Aa  (30%)

  tx_hash  : 0xe09c780fb5d42ed285fbb11bed9733...
  Temps    : 0.517s
  ✅ registerWork() validé


In [3]:
# %% [markdown]
# ## 3. Test verify_registration() et get_work()

# %%
print("Test verify_registration()")
print("─" * 50)

# Hash enregistré
is_registered = manager.verify_registration(hash1)
print(f"  Hash enregistré    : {'✅ OUI' if is_registered else '❌ NON'}")
assert is_registered

# Hash non enregistré
fake_hash = "0" * 64
is_fake   = manager.verify_registration(fake_hash)
print(f"  Hash non enregistré: {'✅ NON' if not is_fake else '❌ OUI (erreur)'}")
assert not is_fake

# %%
print("\nTest get_work()")
print("─" * 50)

work_info = manager.get_work(hash1)
print(f"  CID         : {work_info['cid']}")
print(f"  Recipients  : {work_info['recipients']}")
print(f"  Shares      : {work_info['shares']}")
print(f"  Registered  : {work_info['registered_at']}")

assert work_info["cid"]                  == cid_test
assert work_info["recipients"][0].lower() == recipient1.lower()
assert work_info["shares"]               == [70, 30]
print("  ✅ get_work() validé")

Test verify_registration()
──────────────────────────────────────────────────
  Hash enregistré    : ✅ OUI
  Hash non enregistré: ✅ NON

Test get_work()
──────────────────────────────────────────────────
  CID         : QmTestFingerprint001AAABBBCCC
  Recipients  : ['0x6217386b3d46D78beb721FC3be503f46D2008689', '0xF046B0CBce9EdD632Af48768559DeEB46E2701Aa']
  Shares      : [70, 30]
  Registered  : 1781539916
  ✅ get_work() validé


In [4]:
# %% [markdown]
# ## 4. Test certifyInfringement() — store_proof()

# %%
print("Test store_proof() → certifyInfringement()")
print("─" * 50)

evidence_cid = "QmEvidenceDossierProuveAlterationCID001"

t0 = time.time()
tx_proof, evidence_hash = manager.store_proof(
    evidence_cid = evidence_cid,
    work_hash    = hash1
)
dt = time.time() - t0

print(f"  tx_proof      : {tx_proof[:32]}...")
print(f"  evidence_hash : {evidence_hash[:32]}...")
print(f"  Temps         : {dt:.3f}s")

assert tx_proof.startswith("0x"),     "tx_proof invalide"
assert len(evidence_hash) == 64,      "evidence_hash doit faire 64 chars hex"
assert evidence_hash != "0" * 64,    "evidence_hash ne doit pas être nul"
print("  ✅ store_proof() validé")

# %%
# Double certif sur la même preuve → hash différent (timestamp différent)
_, evidence_hash2 = manager.store_proof(evidence_cid, hash1)
print(f"\n  Double certif → hashes distincts : "
      f"{'✅ OUI' if evidence_hash != evidence_hash2 else '⚠️  IDENTIQUES'}")


Test store_proof() → certifyInfringement()
──────────────────────────────────────────────────
  tx_proof      : 0x86038a86d95ebbf9467fa492606f2c...
  evidence_hash : 6e2f4085dc9a7a0e4680a2efa5584a02...
  Temps         : 0.086s
  ✅ store_proof() validé

  Double certif → hashes distincts : ⚠️  IDENTIQUES


In [6]:
# %% [markdown]
# ## 5. Test simulateRoyaltyPayment()

# %%
print("Test simulateRoyaltyPayment()")
print("─" * 50)

TOTAL_AMOUNT = 1000

t0 = time.time()
tx_royalty, payments = manager.simulate_royalty_payment(
    work_hash    = hash1,
    total_amount = TOTAL_AMOUNT
)
dt = time.time() - t0

print(f"  tx_royalty  : {tx_royalty[:32]}...")
print(f"  Temps       : {dt:.3f}s")
print(f"  Paiements   : {len(payments)}")
print()

total_distributed = 0
for p in payments:
    pct   = p["amount"] / TOTAL_AMOUNT * 100
    total_distributed += p["amount"]
    print(f"  {p['beneficiary'][:20]}... → {p['amount']:6d} unités ({pct:.1f}%)")

print(f"\n  Total distribué : {total_distributed} / {TOTAL_AMOUNT}")
assert len(payments)        == 2,           "2 bénéficiaires attendus"
assert payments[0]["amount"] == 700,         "Artiste A : 70% de 1000 = 700"
assert payments[1]["amount"] == 300,         "Producteur B : 30% de 1000 = 300"
assert total_distributed    == TOTAL_AMOUNT, "La somme doit égaler le total"
print("  ✅ simulateRoyaltyPayment() validé")

Test simulateRoyaltyPayment()
──────────────────────────────────────────────────
  tx_royalty  : 0x6ac16d7d3140f6abb3cdf3aceedfde...
  Temps       : 0.050s
  Paiements   : 2

  0x6217386b3d46D78beb... →    700 unités (70.0%)
  0xF046B0CBce9EdD632A... →    300 unités (30.0%)

  Total distribué : 1000 / 1000
  ✅ simulateRoyaltyPayment() validé


In [7]:
# %% [markdown]
# ## 6. Test gestion des erreurs

# %%
print("Tests gestion des erreurs")
print("─" * 50)

# 1. Enregistrement doublon
try:
    manager.register_work(hash1, cid_test, [recipient1], [100])
    print("  ❌ Doublon : devrait lever une exception")
except Exception as e:
    print(f"  ✅ Doublon bloqué : {str(e)[:60]}")

# 2. Certif sur une œuvre inexistante
try:
    manager.store_proof("QmTest", "0" * 64)
    print("  ❌ Œuvre inexistante : devrait lever une exception")
except Exception as e:
    print(f"  ✅ Œuvre inexistante bloquée : {str(e)[:60]}")

# 3. Simulation sur une œuvre inexistante
try:
    manager.simulate_royalty_payment("0" * 64, 1000)
    print("  ❌ Simulation invalide : devrait lever une exception")
except Exception as e:
    print(f"  ✅ Simulation invalide bloquée : {str(e)[:60]}")

Tests gestion des erreurs
──────────────────────────────────────────────────
  ✅ Doublon bloqué : Transaction échouée : 873a16f2beae4c092b6e11a6c38bc75d670958
  ✅ Œuvre inexistante bloquée : Transaction échouée : 8418576da942213a92b2edc2d34d20e1c36094
  ✅ Simulation invalide bloquée : Transaction échouée : 7e8308118a0da9612d505072283354813cd7e6


In [8]:
# %% [markdown]
# ## 7. Lecture des événements on-chain

# %%
print("\nLecture des événements on-chain")
print("─" * 50)

with open(settings.deployment_json, "r") as f:
    deployment = json.load(f)

contract_addr = Web3.to_checksum_address(deployment["address"])
contract      = w3.eth.contract(address=contract_addr, abi=deployment["abi"])

# Événements WorkRegistered
work_events = contract.events.WorkRegistered().get_logs(from_block=0)
print(f"  WorkRegistered     : {len(work_events)} événement(s)")
for e in work_events:
    wh = e["args"]["workHash"].hex()
    print(f"    → {wh[:32]}...")

# Événements InfringementCertified
infr_events = contract.events.InfringementCertified().get_logs(from_block=0)
print(f"  InfringementCertified : {len(infr_events)} événement(s)")
for e in infr_events:
    print(f"    → evidenceHash: {e['args']['evidenceHash'].hex()[:32]}...")

# Événements PaymentSimulated
pay_events = contract.events.PaymentSimulated().get_logs(from_block=0)
print(f"  PaymentSimulated   : {len(pay_events)} événement(s)")
for e in pay_events:
    print(f"    → {e['args']['beneficiary'][:20]}... "
          f"| {e['args']['amount']} unités")


Lecture des événements on-chain
──────────────────────────────────────────────────
  WorkRegistered     : 1 événement(s)
    → 935856c3274fcd604dc5f89b54a197f5...
  InfringementCertified : 2 événement(s)
    → evidenceHash: 6e2f4085dc9a7a0e4680a2efa5584a02...
    → evidenceHash: 6e2f4085dc9a7a0e4680a2efa5584a02...
  PaymentSimulated   : 2 événement(s)
    → 0x6217386b3d46D78beb... | 700 unités
    → 0xF046B0CBce9EdD632A... | 300 unités


In [9]:
# %% [markdown]
# ## 8. Mesures de coût en gas

# %%
print("\nMesures de coût en gas")
print("─" * 50)

# Enregistrement d'une 2ème œuvre pour mesure propre
audio2  = make_audio_file(freq=880.0)
emb2    = engine.extract_fingerprint(audio2)
hash2   = FingerprintEngine.embedding_to_hash(emb2)

receipt_reg = w3.eth.get_transaction_receipt(
    manager.register_work(hash2, "QmTest002", [accounts[1]], [100])
)
receipt_cert = w3.eth.get_transaction_receipt(
    manager.store_proof("QmEvidence002", hash2)[0]
)
receipt_pay  = w3.eth.get_transaction_receipt(
    manager.simulate_royalty_payment(hash2, 500)[0]
)

gas_price_gwei = 20  # gwei (configurable dans Ganache)

for label, receipt in [
    ("registerWork",            receipt_reg),
    ("certifyInfringement",     receipt_cert),
    ("simulateRoyaltyPayment",  receipt_pay),
]:
    gas  = receipt["gasUsed"]
    cost = gas * gas_price_gwei / 1e9  # en ETH
    print(f"  {label:30s} : {gas:7,} gas  (~{cost:.6f} ETH)")

# Nettoyage
for p in [audio1, audio2]:
    os.unlink(p)


Mesures de coût en gas
──────────────────────────────────────────────────
  registerWork                   : 185,288 gas  (~0.003706 ETH)
  certifyInfringement            :  95,006 gas  (~0.001900 ETH)
  simulateRoyaltyPayment         :  35,554 gas  (~0.000711 ETH)


In [10]:
# %% [markdown]
# ## 9. Résumé

# %%
print("=" * 60)
print("  RÉSUMÉ NOTEBOOK 04 — Intégration Blockchain")
print("=" * 60)
print(f"  registerWork()            : ✅")
print(f"  verify_registration()     : ✅")
print(f"  get_work()                : ✅")
print(f"  store_proof()             : ✅")
print(f"  simulateRoyaltyPayment()  : ✅")
print(f"  Gestion des erreurs       : ✅")
print(f"  Événements on-chain       : ✅")
print(f"  Gas registerWork          : {receipt_reg['gasUsed']:,}")
print("=" * 60)
print("  ✅ Notebook 04 validé — passer au notebook 05")

  RÉSUMÉ NOTEBOOK 04 — Intégration Blockchain
  registerWork()            : ✅
  verify_registration()     : ✅
  get_work()                : ✅
  store_proof()             : ✅
  simulateRoyaltyPayment()  : ✅
  Gestion des erreurs       : ✅
  Événements on-chain       : ✅
  Gas registerWork          : 185,288
  ✅ Notebook 04 validé — passer au notebook 05
